In [1]:
import numpy as np
import pandas as pd
import re

from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
data = pd.read_csv("twitter_dataset.csv")
print(data.head())

   Tweet_ID        Username  \
0         1         julie81   
1         2   richardhester   
2         3  williamsjoseph   
3         4     danielsmary   
4         5      carlwarren   

                                                Text  Retweets  Likes  \
0  Party least receive say or single. Prevent pre...         2     25   
1  Hotel still Congress may member staff. Media d...        35     29   
2  Nice be her debate industry that year. Film wh...        51     25   
3  Laugh explain situation career occur serious. ...        37     18   
4  Involve sense former often approach government...        27     80   

             Timestamp  
0  2023-01-30 11:00:51  
1  2023-01-02 22:45:58  
2  2023-01-18 11:25:19  
3  2023-04-10 22:06:29  
4  2023-01-24 07:12:21  


In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "",text)
    text = re.sub(r"[^a-z\s]", " ", text)

    return text.split()

sentences = data["Text"].apply(clean_text)
print(sentences.head())

0    [party, least, receive, say, or, single, preve...
1    [hotel, still, congress, may, member, staff, m...
2    [nice, be, her, debate, industry, that, year, ...
3    [laugh, explain, situation, career, occur, ser...
4    [involve, sense, former, often, approach, gove...
Name: Text, dtype: object


In [4]:
sentences = [words for words in sentences if len(words)>0]

print("Total Tweets:", len(sentences))

Total Tweets: 10000


In [5]:
model = Word2Vec(
    sentences = sentences,
    vector_size = 100,
    window=5,
    min_count=2,
    sg=0,
    epochs=10
)

In [6]:
def get_vector(words):
    vectors = [model.wv[word] for word in words if word in model.wv]

    if len(vectors)==0:
        return np.zeros(100)

    return np.mean(vectors,axis=0)


documents_vectors = np.array([
    get_vector(words) for words in sentences
])   
print("Document Vectors Shape:", documents_vectors.shape)

Document Vectors Shape: (10000, 100)


In [8]:
user_input = input("Enter your search query:")
query_words = clean_text(user_input)

query_vector = get_vector(query_words)
print(query_vector.shape)

Enter your search query: Party


(100,)


In [9]:
similarity = cosine_similarity(
    [query_vector],
    documents_vectors
)[0]

In [10]:
top_indices = np.argsort(similarity)[::-1][:5]

print("\nRecommended Tweets:\n")

for i in top_indices:
    print("Tweet:", data["Text"].iloc[i])
    print("Similarity:", similarity[i])
    print("-------------------------")


Recommended Tweets:

Tweet: Quickly international show mission thank necessary. Can put minute party able.
Fact series politics team alone we open strategy. Approach arrive prove idea.
Similarity: 0.9857218
-------------------------
Tweet: Step measure there. Event tree truth white. Again set country address.
Accept student individual build. Civil shoulder group alone party.
Area day indicate especially. Lay lead administration sense trip traditional. Politics center down street strong.
Similarity: 0.98505384
-------------------------
Tweet: One for high good lawyer. Speech near question still top.
Relationship including yes able seven agent small. Image any mention student simple statement. Investment magazine at like support.
Similarity: 0.9848375
-------------------------
Tweet: Model management least miss defense particular trouble. Try article tell care summer site.
Indicate grow behavior other language. Occur relationship mention consider party table performance so.
Gas out adul